In [0]:
import os, sys
sys.path.append(os.path.abspath("../src"))
from churn.dbx import get_config
from pyspark.sql import functions as F

cfg = get_config(dbutils)

# Schémas bronze : TOUT en STRING -> aucune ligne perdue à cause d'un mauvais format
SCHEMAS = {
    "clients": "client_id STRING, date_entree_relation STRING, segment STRING, age STRING, region STRING, "
               "nb_produits STRING, has_credit_immo STRING, has_assurance_vie STRING, has_livret STRING, "
               "revenu_mensuel_declare STRING",
    "operations": "operation_id STRING, client_id STRING, date_operation STRING, type_operation STRING, montant STRING",
    "interactions": "interaction_id STRING, client_id STRING, date_interaction STRING, type_interaction STRING, "
                    "motif STRING, satisfaction STRING",
    "clotures": "client_id STRING, date_cloture STRING, motif_cloture STRING",
}

In [0]:
for entity, schema in SCHEMAS.items():          # on traite les 4 tables une par une

    # ---------- 1. LECTURE ----------
    stream = (spark.readStream.format("cloudFiles")                # Auto Loader
              .option("cloudFiles.format", "csv")                   # les fichiers sont des CSV
              .option("header", "true")                             # la 1re ligne contient les noms de colonnes
              .option("cloudFiles.schemaLocation",
                      f"{cfg.checkpoint_path}/schema_{entity}")     # où Auto Loader range ses infos de schéma
              .schema(schema)                                       # on impose nos colonnes (toutes en STRING)
              .load(f"{cfg.landing_path}/{entity}")                 # le DOSSIER à surveiller, ex. landing/operations
              .withColumn("_source_file", F.col("_metadata.file_path"))  # de quel fichier vient la ligne
              .withColumn("_ingested_at", F.current_timestamp()))       # quand on l'a ingérée

    # ---------- 2. ÉCRITURE ----------
    (stream.writeStream
     .option("checkpointLocation",
             f"{cfg.checkpoint_path}/bronze_{entity}")   # la "mémoire" : fichiers déjà traités
     .trigger(availableNow=True)                         # traite ce qui est là, puis s'arrête
     .toTable(cfg.t(f"bronze_{entity}"))                 # écrit dans workspace.attrition_dev.bronze_xxx
     .awaitTermination())                                # attend la fin avant la table suivante

    print(entity, spark.table(cfg.t(f"bronze_{entity}")).count(), "lignes en bronze")

In [0]:
display(spark.table(cfg.t("bronze_operations")).limit(5))